<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1> Módulo 2: Construir Redes Neuronales Recurrentes en TensorFlow / Keras</h1>
    <h3>Aprendizaje Automático Avanzado 2026</h3>
</div>

En este cuaderno de jupyter abordaremos la construcción, entrenamiento y evaluación de modelos basados en **Redes Neuronales Recurrentes (RNN)** utilizando la API de alto nivel de **TensorFlow y Keras**.

Desarrollaremos tres casos de estudio prácticos fundamentales:

1. **Clasificación de Sentimientos en Texto (Dataset IMDB)**:
   - Uso de capas de incrustación de palabras (`layers.Embedding`).
   - Procesamiento bidireccional de secuencias de texto con `layers.Bidirectional(layers.LSTM(...))`.
   - Manejo de longitudes variables de secuencia mediante relleno y truncamiento (*Padding*).

2. **Predicción de Series Temporales (Google Stock Price)**:
   - Construcción de tensores de series temporales tridimensionales `(batch_size, timesteps, features)` mediante el esquema de **ventana deslizante (*Sliding Window*)**.
   - Apilamiento de múltiples capas recurrentes (*Stacked LSTMs*) con `return_sequences=True`.
   - Regularización con `layers.Dropout` y normalización de variables con `MinMaxScaler`.

3. **Generación Autorregresiva de Texto a Nivel de Caracteres (*Char-Level Language Model*)**:
   - Modelado probabilístico secuencial sobre el corpus de obras de Shakespeare.
   - Creación de pipelines eficientes con `tf.data.Dataset`.
   - Algoritmo de muestreo estocástico con **Temperatura ($T$)** para balancear coherencia vs. creatividad.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb

# Fijar semillas para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"Versión de TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs disponibles: {len(gpus)} -> {gpus if gpus else 'Ejecutando en CPU'}")

# 1. Clasificación de Sentimientos con el Dataset IMDB

El dataset de **IMDB** es un punto de referencia estándar en Procesamiento de Lenguaje Natural (PLN). Contiene **50,000 reseñas de películas** altamente polarizadas (25,000 para entrenamiento y 25,000 para prueba), etiquetadas como:
* **`1`**: Sentimiento positivo 
* **`0`**: Sentimiento negativo 

### Flujo de trabajo:
1. **Selección del vocabulario**: Conservamos las $N$ palabras más frecuentes (por ejemplo, las 20,000 más comunes).
2. **Homogeneización (*Padding*)**: Fijamos una longitud máxima de secuencia ($L = 150$ palabras).
3. **Capa Embedding**: Proyecta cada índice entero a un espacio vectorial continuo denso $\mathbb{R}^{d}$.
4. **Capa Recurrente LSTM Bidireccional**: Captura dependencias contextuales hacia adelante y hacia atrás.
5. **Capa Densa Sigmoide**: Produce la probabilidad de sentimiento positivo.

In [ ]:
# Parámetros del dataset
vocab_size = 20000  # Consideramos las 20,000 palabras más frecuentes
max_len = 150       # Longitud fija de las secuencias de texto (número de palabras por reseña)

print("Cargando el dataset de IMDB...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Muestras de prueba:        {len(X_test)}")
print(f"Ejemplo de reseña codificada (primeras 15 palabras): {X_train[0][:15]}")
print(f"Etiqueta de la primera reseña: {y_train[0]} ({'Positiva' if y_train[0] == 1 else 'Negativa'})")

### Exploración y Decodificación de Texto

El dataset de Keras ya viene pre-tokenizado como secuencias de identificadores enteros. Podemos utilizar el diccionario de mapeo `imdb.get_word_index()` para reconstruir y visualizar las reseñas en lenguaje humano:

In [ ]:
word_index = imdb.get_word_index()

# Los primeros índices están reservados en Keras:
# 0: <PAD>, 1: <START>, 2: <UNK>, 3: <UNUSED>
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_text):
    return " ".join([reverse_word_index.get(i, "?") for i in encoded_text])

print("Texto decodificado de la primera reseña:")
print("-" * 70)
print(decode_review(X_train[0][:60]) + " ...")
print("-" * 70)

### Homogeneización de Secuencias (*Padding y Truncating*)

Dado que las reseñas tienen longitudes diferentes y las redes neuronales procesan tensores en lotes (*batches*) de forma matricial fija, debemos estandarizar todas las secuencias a una longitud común de `max_len = 150`:
* Si la reseña es más corta que 150 palabras, se rellena con ceros (`<PAD>`).
* Si la reseña es más larga, se trunca a 150 palabras.

In [ ]:
# Aplicamos pad_sequences
X_train_padded = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_len, padding='post', truncating='post')
X_test_padded = keras.preprocessing.sequence.pad_sequences(X_test, maxlen=max_len, padding='post', truncating='post')

print(f"Forma de X_train_padded: {X_train_padded.shape}")
print(f"Forma de X_test_padded:  {X_test_padded.shape}")
print(f"Primer vector procesado:\n{X_train_padded[0][:25]} ...")

### Construcción de la Red Neuronal Recurrente con LSTM Bidireccional

Diseñaremos una arquitectura robusta para clasificación binaria de texto:
1. `layers.Embedding(input_dim=vocab_size, output_dim=128)`: Aprende una representación vectorial continua para cada palabra.
2. `layers.SpatialDropout1D(0.2)`: Regulariza omitiendo columnas completas de la matriz de incrustación para evitar sobreajuste en palabras específicas.
3. `layers.Bidirectional(layers.LSTM(64, dropout=0.2, recurrent_dropout=0.0))`: Procesa la secuencia en ambas direcciones temporales.
4. `layers.Dense(32, activation='relu')`: Capa completamente conectada para refinar la extracción de patrones.
5. `layers.Dropout(0.3)`: Dropout estándar para generalización.
6. `layers.Dense(1, activation='sigmoid')`: Capa de salida con activación sigmoide para predecir la probabilidad $P(y = 1 \mid \text{texto})$.

In [ ]:
embed_dim = 128


In [ ]:

# Entrenamos por 4 épocas con un subconjunto de validación del 20%


In [ ]:
# Visualización de curvas de aprendizaje
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_imdb.history['loss'], label='Pérdida Entrenamiento', color='royalblue')
plt.plot(history_imdb.history['val_loss'], label='Pérdida Validación', color='crimson', linestyle='--')
plt.title('Evolución de la Pérdida (Binary Crossentropy)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_imdb.history['accuracy'], label='Precisión Entrenamiento', color='royalblue')
plt.plot(history_imdb.history['val_accuracy'], label='Precisión Validación', color='crimson', linestyle='--')
plt.title('Evolución de la Precisión (Accuracy)')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Evaluación sobre el Conjunto de Prueba e Inferencia con Nuevas Reseñas

Evaluamos el modelo final sobre las 25,000 reseñas del conjunto de prueba e implementamos una función `predict_sentiment` para clasificar cualquier texto arbitrario ingresado en inglés:

In [ ]:
# 1. Evaluación en test set
test_loss, test_acc = imdb_model.evaluate(X_test_padded, y_test, batch_size=128, verbose=0)
print(f" Rendimiento en Test Set:")
print(f"   - Pérdida (Loss):    {test_loss:.4f}")
print(f"   - Exactitud (Acc):   {test_acc * 100:.2f}%\n")

# 2. Función de inferencia para frases arbitrarias
def predict_sentiment(text, model, maxlen=max_len):
    # Tokenización simple en minúsculas
    tokens = text.lower().replace('.', ' ').replace(',', ' ').split()
    # Mapeo a índices del vocabulario IMDB
    encoded = [word_index.get(token, 2) + 3 for token in tokens if word_index.get(token, 2) + 3 < vocab_size]
    # Padding
    padded = keras.preprocessing.sequence.pad_sequences([encoded], maxlen=maxlen, padding='post', truncating='post')
    
    # Inferencia
    prob = model.predict(padded, verbose=0)[0][0]
    sentiment = "Positivo" if prob >= 0.5 else "Negativo"
    
    print(f"Reseña: \"{text}\"")
    print(f"Probabilidad Positiva: {prob:.4f} -> Predicción: {sentiment}\n")

# Pruebas con frases de ejemplo
predict_sentiment("This movie was absolutely wonderful, fantastic acting and brilliant plot!", imdb_model)
predict_sentiment("Terrible film, completely waste of time, boring and poorly directed.", imdb_model)
predict_sentiment("The story was okay, some good moments but quite predictable overall.", imdb_model)

# 2. Predicción de Series Temporales Financieras con LSTM Apiladas (Google Stock Price)

En este problema de **regresión secuencial**, el objetivo es pronosticar el precio de apertura (*Open*) de las acciones de Google para el día $t$ utilizando como entrada la ventana temporal de los 60 días de negociación previos ($t-60$ a $t-1$).

### Metodología:
1. **Escalamiento Min-Max**: La función de activación $\tanh$ en las celdas LSTM es altamente sensible a la escala de las entradas. Normalizamos los datos a $[0, 1]$.
2. **Esquema de Ventana Deslizante (*Sliding Window*)**: Transformamos una serie 1D en un tensor tridimensional con forma `(muestras, 60, 1)`.
3. **Red LSTM Profunda (Stacked LSTM)**: Apilamos 4 capas `LSTM` con `Dropout(0.2)` intermedio. Para apilar capas recurrentes, las primeras deben tener activado `return_sequences=True`.
4. **Inversión de Escala y Evaluación**: Revertimos la transformación con `MinMaxScaler.inverse_transform` y contrastamos visualmente con los precios reales del conjunto de test (`Google_Stock_Price_Test.csv`).

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Carga de datos de entrenamiento comprobando rutas relativas
train_csv_path = 'Data/Google_Stock_Price_Train.csv'
dataset_train = pd.read_csv(train_csv_path)

print(f"Dimensiones del dataset de entrenamiento: {dataset_train.shape}")
display(dataset_train.head())

# Extraemos la serie de la columna 'Open'


# Normalización MinMax al rango [0, 1]


In [ ]:
# Parámetros de la ventana temporal

# Construcción de ventanas: desde el índice 60 hasta el final

# Redimensionar a formato 3D requerido por Keras: (muestras, timesteps, features)


### Arquitectura del Modelo Regresor (Stacked LSTM)

Construimos un modelo `Sequential` con 4 capas LSTM profundas:
* **Capas 1, 2 y 3**: `LSTM(units=50, return_sequences=True)` para que cada capa pase su secuencia completa de salidas a la siguiente.
* **Capa 4**: `LSTM(units=50, return_sequences=False)` para obtener únicamente el vector final de características ocultas.
* **Regularización**: `Dropout(0.2)` entre cada bloque LSTM para mitigar el sobreajuste.
* **Capa Densa Final**: `Dense(units=1)` con activación lineal para predecir el valor escalar continuo.

In [ ]:
# Arquitectura


In [ ]:
# Entrenamiento del modelo


In [ ]:
# Gráfica de la función de pérdida durante el entrenamiento
plt.figure(figsize=(8, 4))
plt.plot(history_stock.history['loss'], color='blue', label='Pérdida MSE')
plt.title('Evolución de la Pérdida durante el Entrenamiento (Google Stock LSTM)')
plt.xlabel('Época')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.grid(True)
plt.legend()
plt.show()

### Evaluación sobre el Conjunto de Prueba (`Google_Stock_Price_Test.csv`)

Para predecir el primer día del conjunto de prueba (enero 2017), necesitamos los **60 días inmediatamente anteriores**. Por ello:
1. Concatenamos la columna `Open` del conjunto de entrenamiento y el conjunto de prueba.
2. Tomamos el rango necesario desde `len(total) - len(test) - 60`.
3. Aplicamos `scaler.transform` (usando el escalador ajustado con los datos de train).
4. Generamos las ventanas de entrada para el conjunto de prueba.
5. Invertimos la escala de las predicciones con `scaler.inverse_transform` y graficamos.

In [ ]:
# 1. Carga del conjunto de prueba
test_csv_path = 'Data/Google_Stock_Price_Test.csv' 
dataset_test = pd.read_csv(test_csv_path)
real_stock_price = dataset_test[['Open']].values

In [ ]:
# 2. Concatenación y obtención de las entradas

In [ ]:
# 3. Escalamiento usando el scaler ya entrenado

In [ ]:
# 4. Construcción de ventanas de prueba

In [ ]:
# 5. Inferencia y desescalamiento

In [ ]:
# Visualización comparativa
plt.figure(figsize=(10, 5))
plt.plot(real_stock_price, color='red', label='Precio Real de Google Stock')
plt.plot(predicted_stock_price, color='blue', linestyle='--', label='Precio Predicho por LSTM (TensorFlow)')
plt.title('Predicción del Precio de las Acciones de Google (Enero 2017)')
plt.xlabel('Días Bursátiles de Prueba')
plt.ylabel('Precio de Apertura (USD)')
plt.legend()
plt.grid(True)
plt.show()

# 3. Generación Autorregresiva de Texto a Nivel de Caracteres con LSTM

Un **Modelo de Lenguaje a Nivel de Caracteres (*Character-Level Language Model*)** aprende a predecir el siguiente carácter $c_{t+1}$ condicionada a la secuencia de caracteres observados anteriormente:

$$P(c_{t+1} \mid c_1, c_2, \dots, c_t)$$

<img src="Figures/63.png" alt="Character-level model" style="display: block; margin: 0 auto;" width="500">

### Flujo de Implementación con TensorFlow:
1. **Descarga del Corpus**: Descargamos el texto de las obras de Shakespeare.
2. **Vocabulario y Mapeos**: Mapeamos cada carácter único a un entero (`char2idx`) y viceversa (`idx2char`).
3. **Pipeline con `tf.data.Dataset`**: Dividimos el texto en secuencias de longitud fija `seq_length=100`, donde la entrada es `chunk[:-1]` y el objetivo desplazado es `chunk[1:]`.
4. **Arquitectura LSTM Generativa**: Capa `Embedding`, capas `LSTM` con `return_sequences=True` y capa `Dense(vocab_size)`.
5. **Muestreo con Temperatura ($T$)**: La temperatura controla la dispersión de la distribución de probabilidad durante la generación:
   $$\text{logits}_{\text{ajustados}} = \frac{z_i}{T}$$
   - **$T$ baja (ej. $0.2$)**: Texto muy conservador, repetitivo y gramaticalmente rígido.
   - **$T$ media (ej. $0.7$)**: Buen balance entre fluidez, gramática y creatividad.
   - **$T$ alta (ej. $1.2$)**: Mayor diversidad léxica pero con mayor probabilidad de inventar palabras erróneas.

In [ ]:
# 1. Descarga del corpus de texto
shakespeare_path = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

text = open(shakespeare_path, 'rb').read().decode(encoding='utf-8')
print(f"Longitud total del texto: {len(text):,} caracteres")

# 2. Vocabulario de caracteres únicos
vocab_char = sorted(set(text))
vocab_char_size = len(vocab_char)
print(f"Número de caracteres únicos (vocabulario): {vocab_char_size}")

# Mapeos de caracteres a índices y viceversa
char2idx = {char: idx for idx, char in enumerate(vocab_char)}
idx2char = np.array(vocab_char)

# Representación numérica de todo el texto
text_as_int = np.array([char2idx[c] for c in text])

print("\nPrimeros 100 caracteres del corpus:")
print("-" * 50)
print(text[:100])
print("-" * 50)

In [ ]:
# Longitud máxima de secuencia por muestra y tamaño de lote
SEQ_LEN = 100
BATCH_SIZE = 64
BUFFER_SIZE = 10000

# Función para dividir secuencias en (entrada, objetivo_desplazado_un_caracter)
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


In [ ]:
# Creación del tf.data.Dataset


In [ ]:
# Verificamos un ejemplo de entrada y salida
for input_example, target_example in dataset_gen.take(1):
    print("\nEjemplo Entrada (Input):")
    print(repr("".join(idx2char[input_example[0].numpy()[:40]])))
    print("Ejemplo Salida Esperada (Target):")
    print(repr("".join(idx2char[target_example[0].numpy()[:40]])))

In [ ]:
def build_text_generation_model(vocab_size, embedding_dim=256, rnn_units=512):
    model = keras.Sequential([
        layers.Input(shape=(None,)),  # Acepta secuencias de longitud variable
        layers.Embedding(vocab_size, embedding_dim),
        layers.LSTM(rnn_units, return_sequences=True, recurrent_initializer='glorot_uniform'),
        layers.Dropout(0.2),
        layers.LSTM(rnn_units, return_sequences=True, recurrent_initializer='glorot_uniform'),
        layers.Dropout(0.2),
        layers.Dense(vocab_size)  # Salida de logits (sin softmax para mayor estabilidad numérica)
    ], name="Shakespeare_Char_LSTM")
    return model

char_gen_model = build_text_generation_model(vocab_size=vocab_char_size, embedding_dim=256, rnn_units=512)
char_gen_model.summary()

In [ ]:
# Función de pérdida basada en logits: SparseCategoricalCrossentropy
def char_loss(labels, logits):
    return keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

char_gen_model.compile(optimizer='adam', loss=char_loss)

# Entrenamos por 5 épocas (o sobre un subconjunto de pasos para demostración interactiva rápida)
EPOCHS_GEN = 5

history_gen = char_gen_model.fit(
    dataset_gen.take(250), # 250 lotes por época para balancear velocidad y convergencia
    epochs=EPOCHS_GEN,
    verbose=1
)

### Generación de Texto con Muestreo Estocástico y Temperatura

Para generar texto:
1. Pasamos una cadena de texto inicial (*prompt/seed*).
2. El modelo predice la distribución de probabilidad (logits) para el siguiente carácter.
3. Escalamos los logits por la temperatura ($T$): $\text{logits} = \frac{\text{logits}}{T}$.
4. Muestreamos de forma multinomial usando `tf.random.categorical`.
5. Retroalimentamos el carácter predicho como nueva entrada para generar la secuencia completa paso a paso.

In [ ]:
def generate_text_tf(model, start_string, num_generate=300, temperature=1.0):
    # Vectorización del texto inicial
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    
    text_generated = []
    
    for _ in range(num_generate):
        predictions = model(input_eval, training=False)
        # Tomamos los logits del último paso de tiempo y removemos la dimensión de batch
        predictions = tf.squeeze(predictions, 0)[-1, :]
        
        # Ajuste por temperatura
        predictions = predictions / temperature
        
        # Muestreo de la distribución categórica
        predicted_id = tf.random.categorical(tf.expand_dims(predictions, 0), num_samples=1)[-1, 0].numpy()
        
        # El carácter predicho se agrega al contexto de entrada
        input_eval = tf.concat([input_eval, tf.constant([[predicted_id]], dtype=tf.int32)], axis=-1)
        text_generated.append(idx2char[predicted_id])
        
    return start_string + "".join(text_generated)

# Demostración comparativa con distintas temperaturas
seed_prompt = "ROMEO: "

for temp in [0.3, 0.7, 1.2]:
    print(f"\n{'='*25} TEMPERATURA: {temp} {'='*25}")
    generated = generate_text_tf(char_gen_model, start_string=seed_prompt, num_generate=250, temperature=temp)
    print(generated)